# Vesuvius First Letters: one z band → CUDA spiral fit → every winding → ink_9um

Self-contained and resumable. **Settings:** Accelerator *GPU T4 x2* (or RTX Pro 6000; not L4), **Internet ON**, Persistence off.
Edit the `RUN` cell only. Everything that must survive goes to `/kaggle/working/<tag>/` (montages, per-winding ink TIFs,
fit checkpoint + meshes, logs). To continue an unfinished run in a new session, attach the previous version's output as an
input and set `resume_from` to its `/kaggle/input/...` path.

Modes:
- `control`: PHerc1667 w018 (2.399 µm mesh, known text; it is in ink_9um's training labels, so this checks the stack, not generalisation), rendered at level 2 (9.6 µm).
  No fit. ~15–25 min. Letters must show in **one** direction only.
- `band`: stage official tracks (only the band's byte ranges) + crossings + lasagna rows + umbilicus → `fit_spiral.py`
  on CUDA → render every fitted winding from the 9 µm scan at level 0 → ink_9um seed42+43, both directions → void-masked montages.

In [ ]:
import time; T0 = time.time()
RUN = dict(
    mode="control",            # "control" | "band"
    scroll="PHerc1667",        # e.g. "PHerc0826"
    volume="20251217075048",   # volume id prefix (S3 <scroll>/volumes/<id>-...zarr)
    level=2,                   # pyramid level to render: 2 for 2.4 um scans (-> 9.6 um), 0 for ~9 um scans
    z0=None, z1=None,          # band in level-0 slices; None/"auto" = densest 1000-slice band of published tracks
    steps=3000,                # fit iterations (autosaves every 1000; resumable)
    sense="CW",                # spiral_outward_sense (catalog has no handedness keys; CW worked for PHerc0826/0846A)
    windings=[0, 9999],        # winding index range to render
    ink_mode=None,             # None = "full" for control, "screen" for band. screen: seed42 only, 25% overlap (~4.5x
                               # cheaper, both directions). full: seed42+43 ensemble, 50% overlap (confirm candidates)
    winding_list=None,         # e.g. ["w052","w053"]: only these windings (use with ink_mode="full" + resume_from)
    ink_batch=16,
    hecate=False,              # also run scrollprize/hecate 9.6 um (about 10x slower than ink_9um)
    max_hours=2.8,             # wall budget for this session, incl. setup
    reserve_hours_for_ink=1.0, # stop the fit early (resumable) to leave this much for render+ink
    tag="w018",                # output folder name under /kaggle/working
    resume_from=None,          # e.g. "/kaggle/input/vesuvius-band-pherc0826" (previous output attached)
    float32=True,              # FIT_SPIRAL_MAX_PRECISION_FLOAT=32 (T4 / RTX fp64 is slow)
    triton=True,               # falls back to the eager kernels automatically on a Triton/ptxas failure
    cache_gb=25,
)

In [ ]:
import os, subprocess, sys, json, glob
W = "/kaggle/working"; TMP = "/tmp/vb"; os.makedirs(TMP, exist_ok=True)
VILLA = f"{TMP}/villa"; VFIT = f"{TMP}/venv_fit"; VINK = f"{TMP}/venv_ink"
VILLA_REF = "322cb0899ac9efe2588d5a3395dce6dc2dea3c89"   # commit the local Mac pipeline was validated on
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:], flush=True)
    if check and r.returncode: raise RuntimeError(f"failed: {cmd}")
    return r
sh("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv")
sh("df -h /kaggle/working /tmp | cat; nproc; free -g | head -2")
sh("pip install -q uv")
if not os.path.exists(f"{VILLA}/.git"):
    sh(f"git init -q {VILLA} && cd {VILLA} && git remote add origin https://github.com/ScrollPrize/villa.git "
       f"&& git sparse-checkout set spiral-fitting vc_object_store vesuvius lasagna "
       f"&& (git fetch -q --depth 1 origin {VILLA_REF} && git checkout -q FETCH_HEAD "
       f"|| (git fetch -q --depth 1 origin main && git checkout -q FETCH_HEAD))")
sh(f"cd {VILLA} && git log -1 --format='%H %cd'")
# ink + render env. What ink_9um actually imports (measured on the validated local venv, model built):
# torch, torchvision, timm, einops, huggingface_hub, zarr>=3, numcodecs, pynrrd, imagecodecs, tifffile, numba,
# opencv, scipy, requests/pyyaml/aiohttp. Fast path: reuse Kaggle's preinstalled torch/timm/torchvision
# (venv with system site-packages) and add only the small missing ones; fall back to a full py3.12 venv.
INK_LIGHT = "'zarr>=3,<4' 'numcodecs>=0.15.1,<0.17' pynrrd imagecodecs tifffile einops timm huggingface-hub tqdm requests pyyaml aiohttp fsspec"
INK_CHECK = (f"cd {TMP} && PYTHONPATH={VILLA}/vesuvius/src {VINK}/bin/python -c \"import torch, torchvision, timm, zarr, cv2, numba; "
             f"import vesuvius.ink_detection.inference.infer; assert torch.cuda.is_available() and int(zarr.__version__[0]) >= 3; "
             f"print('ink env ok', torch.__version__, torch.cuda.device_count(), zarr.__version__)\"")
if not os.path.exists(f"{VINK}/bin/python"):
    sh(f"uv venv -q --system-site-packages --python {sys.executable} {VINK}")
    sh(f"uv pip install -q --python {VINK}/bin/python {INK_LIGHT}")
    if sh(INK_CHECK, check=False).returncode:
        print("fast ink env failed its check -> full py3.12 venv", flush=True)
        sh(f"rm -rf {VINK} && uv venv -q --python 3.12 {VINK}")
        sh(f"uv pip install -q --python {VINK}/bin/python torch torchvision --index-url https://download.pytorch.org/whl/cu128")
        sh(f"uv pip install -q --python {VINK}/bin/python {INK_LIGHT} numpy pillow scipy opencv-python-headless numba")
        if sh(INK_CHECK, check=False).returncode:  # driver older than the cu128 runtime: cu126 wheels
            sh(f"uv pip install -q --python {VINK}/bin/python --reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu126")
            sh(INK_CHECK)
# spiral-fitting env (requires python >= 3.14; builds the vc_spiral C++ extension)
if RUN["mode"] == "band" and not os.path.exists(f"{VFIT}/bin/python"):
    sh(f"uv python install -q 3.14 && uv venv -q --python 3.14 {VFIT}")
    sh(f"uv pip install -q --python {VFIT}/bin/python 'torch>=2.11,<2.13' --index-url https://download.pytorch.org/whl/cu128")
    # editable, like the validated local install: a plain install leaves spiral-fitting/vc_spiral/__init__.py (no .so)
    # shadowing the built package when fit_spiral.py runs from its own folder -> no native packed-track loader
    r = sh(f"cd {VILLA}/spiral-fitting && uv pip install -q --python {VFIT}/bin/python -e .", check=False)
    if r.returncode:  # old system g++ without enough C++23: get a newer one and retry
        sh("apt-get -qq update && (apt-get -qq install -y g++-13 || apt-get -qq install -y g++-12)", check=False)
        cxx = "g++-13" if os.path.exists("/usr/bin/g++-13") else "g++-12"
        sh(f"cd {VILLA}/spiral-fitting && CXX={cxx} CC={cxx.replace('++', 'cc')} uv pip install -q --python {VFIT}/bin/python -e .")
CHECK_NATIVE = ("import tracks, vc_spiral.track_store as t, vc_spiral.track_crossings, torch; "
                "assert tracks._load_native_track_store() is not None; print('native ok', t.__file__, torch.__version__, torch.cuda.is_available())")
if RUN["mode"] == "band":
    r = sh(f"cd {VILLA}/spiral-fitting && {VFIT}/bin/python -c \"{CHECK_NATIVE}\"", check=False)
    if r.returncode:  # fall back: put the built extensions next to the source package the fitter's cwd resolves
        so = glob.glob(f"{VFIT}/lib/python3.14/site-packages/vc_spiral/*.so") + glob.glob(f"{VILLA}/spiral-fitting/build/**/vc_spiral/*.so", recursive=True)
        for f in so: sh(f"cp -n {f} {VILLA}/spiral-fitting/vc_spiral/")
        sh(f"cd {VILLA}/spiral-fitting && {VFIT}/bin/python -c \"{CHECK_NATIVE}\"")
# checkpoints
CK = f"{TMP}/ck"
dl = (f"from huggingface_hub import hf_hub_download as h\n"
      f"for f in ['hybrid_3d2d-seed42/step-075000.pth','hybrid_3d2d-seed43/step-075000.pth']: print(h('scrollprize/ink_9um', f, local_dir='{CK}/ink_9um'))\n"
      + (f"for f in ['hecate.py','hecate_9.6um.pth']: print(h('scrollprize/hecate', f, local_dir='{CK}/hecate'))\n" if RUN["hecate"] else ""))
open(f"{TMP}/dl.py", "w").write(dl); sh(f"{VINK}/bin/python {TMP}/dl.py")
smoke = (f"import sys, numpy as np; sys.path.insert(0, '{TMP}/src'); from ink import Ink; from render_gpu import write_zarr\n"
         f"if __name__ == '__main__':\n"
         f" write_zarr('{TMP}/smoke.zarr', np.random.default_rng(0).integers(60, 200, (31, 256, 256), dtype=np.uint8), {{}})\n"
         f" ink = Ink('{CK}/ink_9um/hybrid_3d2d-seed42/step-075000.pth', ['{CK}/ink_9um/hybrid_3d2d-seed43/step-075000.pth'], amp='fp16', batch=4)\n"
         f" print('ink smoke', ink('{TMP}/smoke.zarr', '{TMP}/smoke.tif'))\n")
SMOKE = smoke
print(f"setup done in {(time.time() - T0) / 60:.1f} min")

In [ ]:
os.makedirs('/tmp/vb/src', exist_ok=True)

In [ ]:
%%writefile /tmp/vb/src/umbilicus_est.py
"""Estimate a scroll's umbilicus from lasagna normals: per z slice, the point closest (weighted
least squares) to all lines through sheet pixels along their in-plane normal. Sheet normals of a
rolled scroll point (roughly) at the scroll axis, so these lines meet near the umbilicus.

    estimate(read_slice, zs, scale) -> [{"x","y","z"}...] in level-0 voxels
read_slice(z_lo_res) -> (nx_u8, ny_u8, grad_mag_u8) 2D arrays of one lasagna slice (group grid)."""
import numpy as np


def slice_centre(nx, ny, gm, prev=None, iters=6):
    n = np.stack([(nx.astype(np.float32) - 128) / 127, (ny.astype(np.float32) - 128) / 127], -1)
    mag = np.linalg.norm(n, axis=-1)
    w = gm.astype(np.float32) * (mag > 0.3) * ((nx > 0) | (ny > 0))
    ys, xs = np.nonzero(w > np.percentile(w[w > 0], 50) if (w > 0).any() else w > 0)
    if len(ys) < 200:
        return None, 0.0
    if len(ys) > 200000:
        sel = np.random.default_rng(0).choice(len(ys), 200000, replace=False); ys, xs = ys[sel], xs[sel]
    u = n[ys, xs] / mag[ys, xs, None]
    p = np.stack([xs, ys], -1).astype(np.float64)
    wt = w[ys, xs].astype(np.float64)
    c = p.mean(0) if prev is None else np.asarray(prev, np.float64)
    for it in range(iters):
        P = np.eye(2)[None] - u[:, :, None] * u[:, None, :]          # projector off the normal
        r = np.einsum('nij,nj->ni', P, c[None] - p)
        d = np.linalg.norm(r, axis=-1)
        if it:  # Cauchy-reweight: ignore lines that pass far from the current centre (folds, cracks)
            s = np.median(d) + 1e-6
            ww = wt / (1 + (d / s) ** 2)
        else:
            ww = wt
        A = (ww[:, None, None] * P).sum(0); b = (ww[:, None, None] * P @ p[:, :, None])[:, :, 0].sum(0)
        c = np.linalg.solve(A + 1e-6 * np.eye(2), b)
    ev = np.linalg.eigvalsh(A)
    return c, float(ev[0] / (ev[1] + 1e-9))  # conditioning: ~0 when all normals are parallel (crushed)


def estimate(read_slice, zs_lo, scale, smooth=5):
    pts, prev = [], None
    for z in zs_lo:
        nx, ny, gm = read_slice(z)
        c, cond = slice_centre(nx, ny, gm, prev)
        if c is None:
            continue
        pts.append((z, c[0], c[1], cond)); prev = c
    a = np.array(pts)
    if len(a) >= smooth:  # running median over z
        k = smooth // 2
        a[:, 1] = [np.median(a[max(0, i - k):i + k + 1, 1]) for i in range(len(a))]
        a[:, 2] = [np.median(a[max(0, i - k):i + k + 1, 2]) for i in range(len(a))]
    return [{"x": float(x * scale), "y": float(y * scale), "z": float(z * scale), "cond": round(c, 3)}
            for z, x, y, c in a]

In [ ]:
%%writefile /tmp/vb/src/fetch.py
"""Stage a spiral-fit dataset for one z band without bulk downloads.

- .vctracks index arrays: whole (a few hundred MB); coordinates.i32: sparse file holding only the
  byte ranges of tracks lying wholly inside [z0, z1) (the fit's packed-store loader keeps exactly
  those tracks, so the rest is never read);
- the .dbm SQLite: a sparse stub (real 4 KB header, original size and mtime) so the packed store's
  and the crossings cache's `source_db_signature` match without the 5-12 GB file;
- crossings .npz: whole;
- lasagna nx/ny/grad_mag: only the z chunk rows of the band, at the 4x group;
- umbilicus: published one if present, else estimated (lasagna normals, else CT mask centroid).
"""
import json, os, re, time, urllib.request, urllib.error, concurrent.futures as cf
import numpy as np

DL = "https://dl.ash2txt.org/datasets/spiral_datasets"
S3 = "https://vesuvius-challenge-open-data.s3.amazonaws.com"


def http(url, rng=None, timeout=120, tries=8):
    err = None
    for a in range(tries):
        try:
            req = urllib.request.Request(url, headers={"Range": f"bytes={rng[0]}-{rng[1] - 1}"} if rng else {})
            return urllib.request.urlopen(req, timeout=timeout).read()
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return None
            err = e
        except Exception as e:
            err = e
        time.sleep(min(30, 2 * (a + 1)))
    raise RuntimeError(f"{url} {rng}: {err}")


def head_size(url):
    req = urllib.request.Request(url, method="HEAD")
    return int(urllib.request.urlopen(req, timeout=60).headers["Content-Length"])


def download(url, path, threads=16, part=64 << 20):
    """Parallel ranged download of one big file (resumable per part)."""
    if os.path.exists(path):
        return path
    size = head_size(url)
    tmp = path + ".part"
    with open(tmp, "ab") as f:
        f.truncate(size)
    done_p = tmp + ".done"
    done = set(json.load(open(done_p))) if os.path.exists(done_p) else set()
    fd = os.open(tmp, os.O_WRONLY)
    parts = [(s, min(size, s + part)) for s in range(0, size, part) if s not in done]

    def get(r):
        b = http(url, r)
        assert b is not None and len(b) == r[1] - r[0], f"short read {r}"
        os.pwrite(fd, b, r[0])
        return r[0]
    with cf.ThreadPoolExecutor(threads) as ex:
        for i, s in enumerate(ex.map(get, parts)):
            done.add(s)
            if i % 8 == 0:
                json.dump(sorted(done), open(done_p, "w"))
                print(f"  {os.path.basename(path)} {len(done) * part / 1e9:.1f}/{size / 1e9:.1f} GB", flush=True)
    os.close(fd)
    os.replace(tmp, path)
    if os.path.exists(done_p):
        os.remove(done_p)
    return path


def s3_list(prefix):
    out_p, out_k, token = [], [], None
    while True:
        u = f"{S3}/?list-type=2&delimiter=/&prefix={prefix}" + (f"&continuation-token={urllib.request.quote(token)}" if token else "")
        h = http(u).decode()
        out_p += re.findall(r"<Prefix>([^<]+)</Prefix>", h)[1:]
        out_k += re.findall(r"<Key>([^<]+)</Key>", h)
        m = re.search(r"<NextContinuationToken>([^<]+)<", h)
        if not m:
            return out_p, out_k
        token = m.group(1)


def resolve_volume(scroll, volume):
    """'20250821151701' or a full name -> (full zarr name, voxel size um)."""
    vols = [p.rstrip("/").split("/")[-1] for p in s3_list(f"{scroll}/volumes/")[0]]
    m = [v for v in vols if v.startswith(str(volume))]
    if not m:
        raise SystemExit(f"volume {volume} not in {vols}")
    return m[0], float(re.search(r"-([\d.]+)um-", m[0]).group(1))


def stage_tracks(scroll, vol_ts, ds):
    base = f"{DL}/{scroll}/{vol_ts}/tracks/"
    names = re.findall(r'href="([^"?]+)"', http(base).decode())
    dbm = next(n for n in names if n.endswith(".dbm"))
    crossings = dbm + ".crossings.npz"
    store = os.path.join(ds, "tracks", dbm + ".vctracks")
    os.makedirs(store, exist_ok=True)
    for f in ("header.bin", "metadata.json", "z_bounds.i32", "offsets.i64", "source_ids.u64",
              "arclengths.f64", "tortuosities.f64", "family_codes.i8"):
        download(f"{base}{dbm}.vctracks/{f}", os.path.join(store, f), threads=8, part=16 << 20)
    meta = json.load(open(os.path.join(store, "metadata.json")))
    _, size, mtime_ns = meta["source_db_signature"][0]
    stub = os.path.join(ds, "tracks", dbm)
    if not os.path.exists(stub):
        hdr = http(base + dbm, (0, 4096))
        with open(stub, "wb") as f:
            f.write(hdr); f.truncate(size)
        os.utime(stub, ns=(mtime_ns, mtime_ns))
    download(base + crossings, os.path.join(ds, "tracks", crossings), threads=16)
    return store, dbm


def auto_band(store, height=1000, step=250):
    zb = np.fromfile(f"{store}/z_bounds.i32", "<i4").reshape(-1, 2)
    off = np.fromfile(f"{store}/offsets.i64", "<i8")
    n = np.diff(off)
    best = None
    for z0 in range(int(zb[:, 0].min()) // step * step, int(zb[:, 1].max()) - height, step):
        s = int(n[(zb[:, 0] >= z0) & (zb[:, 1] < z0 + height)].sum())
        if best is None or s > best[1]:
            best = (z0, s)
    return best[0], best[0] + height, best[1]


def stage_band_coords(store, dbm_url, z0, z1, gap=512 * 1024, threads=24):
    """Sparse coordinates.i32 with only the band's tracks (resumable: range list is recorded)."""
    url = dbm_url + ".vctracks/coordinates.i32"
    zb = np.fromfile(f"{store}/z_bounds.i32", "<i4").reshape(-1, 2)
    off = np.fromfile(f"{store}/offsets.i64", "<i8")
    idx = np.nonzero((zb[:, 0] >= z0) & (zb[:, 1] < z1))[0]
    ranges = []
    for s, e in zip(off[idx] * 12, off[idx + 1] * 12):
        if ranges and s - ranges[-1][1] <= gap:
            ranges[-1][1] = max(ranges[-1][1], int(e))
        else:
            ranges.append([int(s), int(e)])
    path = f"{store}/coordinates.i32"
    mark = f"{path}.band_{z0}_{z1}"
    if os.path.exists(mark):
        return len(idx)
    print(f"  band [{z0},{z1}): {len(idx)} tracks, {len(ranges)} ranges, "
          f"{sum(e - s for s, e in ranges) / 1e9:.2f} GB", flush=True)
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.truncate(int(off[-1]) * 12)
    fd = os.open(path, os.O_WRONLY)

    def get(r):
        b = http(url, tuple(r))
        assert b is not None and len(b) == r[1] - r[0]
        os.pwrite(fd, b, r[0])
        return len(b)
    tot = 0
    with cf.ThreadPoolExecutor(threads) as ex:
        for i, n in enumerate(ex.map(get, ranges)):
            tot += n
            if i % 1000 == 0:
                print(f"  {i}/{len(ranges)} {tot / 1e9:.2f} GB", flush=True)
    os.close(fd)
    open(mark, "w").write("ok")
    return len(idx)


def stage_lasagna(scroll, vol_ts, ds, z0, z1, want_scale=4):
    """Mirror the band's z rows of nx/ny/grad_mag. Returns (group, scale, stores) or None."""
    pre = f"{scroll}/representations/predictions/lasagna/"
    runs = [p for p in s3_list(pre)[0] if p.rstrip("/").split("/")[-1].startswith(vol_ts)]
    if not runs:
        return None
    run = sorted(runs)[-1]
    stores = {}
    for p in s3_list(run)[0]:
        name = p.rstrip("/").split("/")[-1]
        for k in ("nx", "ny", "grad_mag"):
            if name.endswith(f"_{k}.ome.zarr"):
                stores[k] = (p, name)
    if set(stores) != {"nx", "ny", "grad_mag"}:
        return None
    za = json.loads(http(f"{S3}/{stores['nx'][0]}.zattrs"))
    group, scale = None, None
    for d in za["multiscales"][0]["datasets"]:
        sc = d["coordinateTransformations"][0]["scale"][0]
        if scale is None or abs(sc - want_scale) < abs(scale - want_scale):
            group, scale = d["path"], int(round(sc))
    out_dir = os.path.join(ds, "lasagna_inputs")
    jobs = []
    for k, (p, name) in stores.items():
        out = os.path.join(out_dir, name)
        for rel in (".zattrs", ".zgroup", f"{group}/.zarray", f"{group}/.zattrs"):
            fp = os.path.join(out, rel)
            if not os.path.exists(fp):
                b = http(f"{S3}/{p}{rel}")
                if b is not None:
                    os.makedirs(os.path.dirname(fp), exist_ok=True); open(fp, "wb").write(b)
        a = json.load(open(os.path.join(out, group, ".zarray")))
        cz, cy, cx = a["chunks"]; Z, Y, X = a["shape"]
        zr = range(max(0, (z0 // scale) // cz - 1), min(-(-(z1 // scale) // cz) + 1, -(-Z // cz)))
        jobs += [(f"{S3}/{p}{group}/{i}/{j}/{l}", os.path.join(out, group, str(i), str(j), str(l)))
                 for i in zr for j in range(-(-Y // cy)) for l in range(-(-X // cx))]

    def fetch(job):
        u, fp = job
        if os.path.exists(fp):
            return 1
        b = http(u, timeout=60)
        if b is None:
            return 0
        os.makedirs(os.path.dirname(fp), exist_ok=True)
        open(fp + ".tmp", "wb").write(b); os.replace(fp + ".tmp", fp)
        return 1
    print(f"  lasagna group {group} (x{scale}): {len(jobs)} chunks", flush=True)
    n = 0
    with cf.ThreadPoolExecutor(64) as ex:
        for i, r in enumerate(ex.map(fetch, jobs)):
            n += r
            if i % 20000 == 0:
                print(f"  {i}/{len(jobs)} ({n} present)", flush=True)
    return group, scale, {k: os.path.join(out_dir, v[1]) for k, v in stores.items()}


def published_umbilicus(scroll, vol_ts):
    keys = [k for k in s3_list(f"{scroll}/representations/umbilicus/")[1]
            if k.split("/")[-1].startswith(vol_ts) and k.endswith(".json")]
    if not keys:
        return None
    return json.loads(http(f"{S3}/{sorted(keys)[-1]}"))


def lasagna_umbilicus(stores, group, scale, z0, z1):
    import zarr
    from umbilicus_est import estimate
    nx, ny, gm = [zarr.open_array(os.path.join(stores[k], group), mode="r") for k in ("nx", "ny", "grad_mag")]
    zs = list(range(z0 // scale, z1 // scale, max(1, 64 // scale)))
    # lasagna nx/ny: the stored "nx" behaves as the row (y) component in the zarr's (z, y, x) grid;
    # (ny, nx) put the estimate within ~500 px of PHerc0826's published umbilicus (crushed scroll),
    # (nx, ny) was ~2000 px off.
    return estimate(lambda z: (ny[z], nx[z], gm[z]), zs, scale)


def ct_umbilicus(vol_url, z0, z1, level=3):
    """Fallback: intensity-mask centroid of a few low-res CT slices."""
    from render_gpu import RemoteVolume
    v = RemoteVolume(vol_url, level)
    s = 2 ** level
    pts = []
    pool = cf.ThreadPoolExecutor(32)
    for z in np.linspace(z0, z1 - 1, 5):
        zl = int(z) // s
        sl, _ = v.read((zl, 0, 0), (zl + 1, v.shape[1], v.shape[2]), pool)
        a = sl[0].astype(np.float32)
        if not (a > 0).any():
            continue
        m = a > np.percentile(a[a > 0], 60)
        ys, xs = np.nonzero(m)
        pts.append({"x": float(xs.mean() * s), "y": float(ys.mean() * s), "z": float(z)})
    return pts


def write_spec(ds, scroll, voxel, sense, dbm, lasagna):
    spec = {"schema_version": 1, "name": scroll, "voxel_size_um": voxel, "spiral_outward_sense": sense,
            "paths": {"tracks_dbm": f"tracks/{dbm}"}}
    if lasagna:
        group, scale, stores = lasagna
        spec.update(normal_zarr_group=group, lasagna_scale=scale)
        spec["paths"].update(normal_x=os.path.relpath(stores["nx"], ds), normal_y=os.path.relpath(stores["ny"], ds),
                             gradient_magnitude=os.path.relpath(stores["grad_mag"], ds))
    json.dump(spec, open(os.path.join(ds, "spiral-scroll.json"), "w"), indent=1)
    return spec

In [ ]:
%%writefile /tmp/vb/src/render_gpu.py
"""Render a tifxyz surface into a flat surface-volume zarr, streaming the scroll volume from the
open-data bucket (uncompressed 128^3 zarr v2 chunks). Port of ../render.py (validated r=0.968
against an official surface volume): same geometry, layer convention and output layout, but
  - every chunk a winding needs is prefetched in parallel before sampling,
  - trilinear sampling runs on the GPU (torch grid_sample, align_corners=True == map_coordinates order=1).
"""
import json, os, sys, threading, time, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor
import numpy as np, tifffile, cv2

CACHE = os.environ.get("CHUNK_CACHE", "/kaggle/tmp/chunks")


class RemoteVolume:
    def __init__(self, url, level=0, cache=None):
        self.url = f"{url.rstrip('/')}/{level}"
        meta = json.loads(urllib.request.urlopen(self.url + "/.zarray", timeout=60).read())
        assert meta["compressor"] is None and meta["dtype"] == "|u1", meta
        self.shape, self.cs = meta["shape"], meta["chunks"][0]
        key = url.rstrip("/").split("/")
        self.cache = os.path.join(cache or CACHE, key[-3], key[-1], str(level))

    def path(self, cz, cy, cx):
        return os.path.join(self.cache, f"{cz}/{cy}/{cx}")

    def fetch(self, cz, cy, cx):
        p = self.path(cz, cy, cx)
        if os.path.exists(p):
            return
        raw = b""
        for attempt in range(8):
            try:
                raw = urllib.request.urlopen(f"{self.url}/{cz}/{cy}/{cx}", timeout=60).read()
                break
            except urllib.error.HTTPError as e:
                if e.code in (403, 404):
                    break  # masked / absent chunk = fill 0
                if attempt == 7:
                    raise
            except Exception:
                if attempt == 7:
                    raise
            time.sleep(1 + attempt)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        tmp = f"{p}.{threading.get_ident()}.part"
        open(tmp, "wb").write(raw)
        os.replace(tmp, p)

    def chunk(self, cz, cy, cx):
        self.fetch(cz, cy, cx)
        raw = open(self.path(cz, cy, cx), "rb").read()
        c = self.cs
        if not raw:
            return None
        return np.frombuffer(raw, np.uint8).reshape(c, c, c)

    def keys(self, lo, hi):
        lo = [max(0, int(v)) for v in lo]
        hi = [min(s, int(v)) for v, s in zip(hi, self.shape)]
        if any(h <= l for l, h in zip(lo, hi)):
            return []
        c = self.cs
        return [(z, y, x) for z in range(lo[0] // c, (hi[0] - 1) // c + 1)
                for y in range(lo[1] // c, (hi[1] - 1) // c + 1)
                for x in range(lo[2] // c, (hi[2] - 1) // c + 1)]

    def read(self, lo, hi, pool):
        """Dense crop [lo, hi) in (z, y, x) voxels, zeros outside."""
        lo = [max(0, int(v)) for v in lo]
        hi = [min(s, int(v)) for v, s in zip(hi, self.shape)]
        out = np.zeros([max(0, h - l) for l, h in zip(lo, hi)], np.uint8)
        if out.size == 0:
            return out, lo
        c = self.cs
        idx = self.keys(lo, hi)
        for (z, y, x), ch in zip(idx, pool.map(lambda t: self.chunk(*t), idx)):
            if ch is None:
                continue
            sl_o, sl_c = [], []
            for a, b in zip((z, y, x), range(3)):
                s0, s1 = max(lo[b], a * c), min(hi[b], (a + 1) * c)
                sl_o.append(slice(s0 - lo[b], s1 - lo[b]))
                sl_c.append(slice(s0 - a * c, s1 - a * c))
            out[tuple(sl_o)] = ch[tuple(sl_c)]
        return out, lo


def load_tifxyz(d, step=None):
    xyz = np.stack([tifffile.imread(os.path.join(d, f"{a}.tif")).astype(np.float32) for a in "xyz"], -1)
    meta = json.load(open(os.path.join(d, "meta.json")))
    valid = (xyz[..., 2] > 0) & np.isfinite(xyz).all(-1)
    up = step or round(1 / meta["scale"][0])
    H, W = xyz.shape[:2]
    # output pixel j <-> mesh coordinate j / up (the published surface volumes'
    # grid); cv2.resize's pixel-centre convention would shift by (up - 1) / 2 px
    gy, gx = np.meshgrid(np.arange(H * up, dtype=np.float32) / up,
                         np.arange(W * up, dtype=np.float32) / up, indexing="ij")
    big = cv2.remap(np.where(valid[..., None], xyz, 0), gx, gy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    vm = cv2.remap(valid.astype(np.float32), gx, gy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT) > 0.999
    return big, vm


def normals(p):
    du = np.gradient(p, axis=1)
    dv = np.gradient(p, axis=0)
    n = np.cross(du, dv)
    n /= np.linalg.norm(n, axis=-1, keepdims=True) + 1e-9
    return n


def sample_trilinear(crop, zyx, device):
    """crop (D,H,W) uint8, zyx (M,3) float voxel coords into crop -> (M,) float32, 0 outside."""
    import torch
    D, H, W = crop.shape
    vol = torch.from_numpy(np.ascontiguousarray(crop)).to(device, non_blocking=True).float()[None, None]
    g = torch.from_numpy(np.ascontiguousarray(zyx, np.float32)).to(device)
    size = torch.tensor([W - 1, H - 1, D - 1], device=device, dtype=torch.float32).clamp_min(1)
    grid = (g[:, [2, 1, 0]] / size * 2 - 1).view(1, 1, 1, -1, 3)
    out = torch.nn.functional.grid_sample(vol, grid, mode="bilinear", padding_mode="zeros", align_corners=True)
    return out.view(-1).cpu().numpy()


def render(tifxyz, vol_url, out, layers=31, level=0, tile=256, flip=False, frame_level=0,
           device="cpu", threads=64, log=print):
    """Returns (canvas [L,H,W] uint8, valid mask). Writes an uncompressed zarr v2 at `out`."""
    sc = 2 ** (level - frame_level)
    meta = json.load(open(os.path.join(tifxyz, "meta.json")))
    pts, vm = load_tifxyz(tifxyz, step=max(1, round(1 / meta["scale"][0] / sc)))
    n = normals(pts) * (-1 if flip else 1)
    vol = RemoteVolume(vol_url, level)
    H, W = vm.shape
    half = layers // 2
    offs = (np.arange(layers) - (layers - 1) / 2).astype(np.float32)
    tiles = []
    for y in range(0, H, tile):
        for x in range(0, W, tile):
            m = vm[y:y + tile, x:x + tile]
            if not m.any():
                continue
            p = pts[y:y + tile, x:x + tile][m] / sc
            zyx_c = p[:, ::-1]
            ext = half + 2
            lo = np.floor(zyx_c.min(0)) - ext
            hi = np.ceil(zyx_c.max(0)) + ext + 1
            tiles.append((y, x, lo, hi))
    keys = sorted({k for _, _, lo, hi in tiles for k in vol.keys(lo, hi)})
    t0 = time.time()
    with ThreadPoolExecutor(threads) as ex:
        list(ex.map(lambda k: vol.fetch(*k), keys))
    log(f"   prefetched {len(keys)} chunks in {time.time() - t0:.0f}s; {len(tiles)} tiles {H}x{W}")
    canvas = np.zeros((layers, H, W), np.uint8)
    pool = ThreadPoolExecutor(16)
    for i, (y, x, _, _) in enumerate(tiles):
        m = vm[y:y + tile, x:x + tile]
        p = pts[y:y + tile, x:x + tile][m] / sc
        nn = n[y:y + tile, x:x + tile][m]
        q = p[None] + offs[:, None, None] * nn[None]
        zyx = q[..., ::-1].reshape(-1, 3)
        lo = np.floor(zyx.min(0)) - 1
        hi = np.ceil(zyx.max(0)) + 2
        crop, o = vol.read(lo, hi, pool)
        if crop.size == 0 or not crop.any():
            continue
        v = sample_trilinear(crop, zyx - np.array(o, np.float32), device).reshape(layers, -1)
        blk = canvas[:, y:y + m.shape[0], x:x + m.shape[1]]
        blk[:, m] = np.clip(np.rint(v), 0, 255).astype(np.uint8)
    pool.shutdown()
    write_zarr(out, canvas, {"source_tifxyz": os.path.abspath(tifxyz), "volume": vol_url, "level": level, "flip": flip})
    return canvas, vm


def write_zarr(out, canvas, attrs):
    layers, H, W = canvas.shape
    os.makedirs(os.path.join(out, "0"), exist_ok=True)
    json.dump({"zarr_format": 2}, open(os.path.join(out, ".zgroup"), "w"))
    json.dump({"multiscales": [{"version": "0.4", "axes": [{"name": a, "type": "space"} for a in "zyx"],
               "datasets": [{"path": "0", "coordinateTransformations": [{"type": "scale", "scale": [1, 1, 1]}]}]}],
               **attrs}, open(os.path.join(out, ".zattrs"), "w"))
    json.dump({"zarr_format": 2, "shape": [layers, H, W], "chunks": [layers, 128, 128], "dtype": "|u1",
               "compressor": None, "fill_value": 0, "order": "C", "filters": None, "dimension_separator": "/"},
              open(os.path.join(out, "0", ".zarray"), "w"))
    for cy in range(0, (H + 127) // 128):
        for cx in range(0, (W + 127) // 128):
            s = canvas[:, cy * 128:(cy + 1) * 128, cx * 128:(cx + 1) * 128]
            if not s.any():
                continue
            b = np.zeros((layers, 128, 128), np.uint8)
            b[:, :s.shape[1], :s.shape[2]] = s
            d = os.path.join(out, "0", "0", str(cy))
            os.makedirs(d, exist_ok=True)
            b.tofile(os.path.join(d, str(cx)))


def prune_cache(root, max_gb):
    files = []
    for dp, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(dp, f)
            try:
                st = os.stat(p)
            except FileNotFoundError:
                continue
            files.append((st.st_mtime, st.st_size, p))
    tot = sum(s for _, s, _ in files)
    for _, s, p in sorted(files):
        if tot <= max_gb * 1e9:
            break
        os.remove(p); tot -= s
    return tot

In [ ]:
%%writefile /tmp/vb/src/montage.py
"""Stack winding ink predictions (one direction) into tall review sheets, outer windings first.
Port of ../montage.py with explicit paths: void/crack rims light up the ink model, so voids
(dark in the surface max-composite) are blanked with a 25 px margin before review."""
import glob, os, re
import cv2, numpy as np, tifffile


def norm(a):
    a = a.astype(np.float32); nz = a[a > 0]
    if not nz.size:
        return np.zeros(a.shape, np.uint8)
    lo, hi = np.percentile(nz, 1), np.percentile(nz, 99.7)
    return (np.clip((a - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)


def void_mask(max_png, shape):
    if not os.path.exists(max_png):
        return np.ones(shape, bool)
    s = cv2.imread(max_png, 0)[:shape[0], :shape[1]].astype(np.float32)
    if not (s > 0).any():
        return np.ones(shape, bool)
    void = (s < 0.4 * np.median(s[s > 0])).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(void)
    keep = np.zeros(n, np.uint8); keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= 400  # real voids, not fibre gaps
    void = cv2.dilate(keep[lab], cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25)))
    m = np.ones(shape, bool); m[:s.shape[0], :s.shape[1]] = void == 0
    return m


def winding_no(f):
    return int(re.search(r"w(\d+)", os.path.basename(f)).group(1))


def montage(pred_dir, render_dir, direction, out_prefix, width=3000, per=6, suffix=".tif"):
    fs = sorted(f for f in glob.glob(f"{pred_dir}/w*{suffix}")
                if f.endswith(f"_reverse{suffix}") == (direction == "reverse"))
    fs.sort(key=lambda f: -winding_no(f))
    outs = []
    for i in range(0, len(fs), per):
        rows = []
        for f in fs[i:i + per]:
            raw = tifffile.imread(f)
            name = re.search(r"w\d+", os.path.basename(f)).group(0)
            a = norm(raw) * void_mask(os.path.join(render_dir, f"{name}_max.png"), raw.shape)
            s = width / a.shape[1]
            a = cv2.resize(a, (width, max(1, int(a.shape[0] * s))), interpolation=cv2.INTER_AREA)
            cv2.putText(a, name, (10, 40), 0, 1.3, 255, 3); rows.append(a)
        p = f"{out_prefix}_{i // per}.jpg"
        cv2.imwrite(p, np.vstack(rows), [cv2.IMWRITE_JPEG_QUALITY, 90])
        outs.append(p)
    return outs


def masked_fullres(pred_tif, max_png, out_jpg):
    """Full-resolution masked JPG of one prediction, for reading letters at 1:1."""
    raw = tifffile.imread(pred_tif)
    cv2.imwrite(out_jpg, norm(raw) * void_mask(max_png, raw.shape), [cv2.IMWRITE_JPEG_QUALITY, 88])

In [ ]:
%%writefile /tmp/vb/src/ink.py
"""In-process ink_9um inference (model loaded once, many windings): same entry points and flags as
`python -m vesuvius.ink_detection.inference.infer <zarr> <ck> <out> --overlap 0.5 --blend-mode hann
--direction both --no-compile --ensemble-checkpoint <ck2>` in ../run_segments.sh."""
import os, subprocess, sys
from pathlib import Path
import numpy as np, tifffile


class Ink:
    def __init__(self, ck, ensemble=(), gpus="", amp="auto", batch=16, workers=4, overlap=0.5):
        from vesuvius.ink_detection.inference import infer as I
        from vesuvius.ink_detection.inference.inference_runtime import prepare_model_for_inference
        argv = ["in.zarr", str(ck), "out.tif", "--overlap", str(overlap), "--blend-mode", "hann", "--batch-size", str(batch),
                "--direction", "both", "--no-compile", "--amp-dtype", amp, "--num-workers", str(workers)]
        for e in ensemble:
            argv += ["--ensemble-checkpoint", str(e)]
        if gpus:
            argv += ["--gpus", gpus]
        self.I = I
        self.args = I.normalize_inference_paths(I.parse_args(argv))
        cm = I.configure_model(self.args)
        model, self.device = prepare_model_for_inference(cm.model, gpu_ids=self.args.gpu_ids, compile_model=False,
                                                         compile_mode=self.args.compile_mode)
        from dataclasses import replace
        self.cm = replace(cm, model=model)

    def __call__(self, zarr_path, out_tif):
        """Writes out_tif (forward) and <stem>_reverse.tif, zlib-compressed."""
        out_tif = Path(out_tif)
        outs = []
        for d in ("forward", "reverse"):
            o = out_tif if d == "forward" else out_tif.with_name(f"{out_tif.stem}_reverse{out_tif.suffix}")
            tmp = o.with_name(o.stem + ".raw.tif")
            self.args.input_zarr = Path(zarr_path)
            self.I.infer_single_zarr(args=self.args, input_zarr=Path(zarr_path), configured_model=self.cm,
                                     device=self.device, output_tiff=tmp, layer_direction=d)
            tifffile.imwrite(o, tifffile.imread(tmp), compression="zlib")
            tmp.unlink()
            outs.append(o)
        return outs


def hecate(hecate_py, ck, zarr_path, out_png, spacing_um, reverse=False, device="cuda"):
    cmd = [sys.executable, str(hecate_py), "--checkpoint", str(ck), "--input", str(zarr_path), "--output", str(out_png),
           "--spacing-um", str(spacing_um), "--device", device, "--batch-size", "8"]
    if reverse:
        cmd.append("--reverse")
    return subprocess.run(cmd, capture_output=True, text=True)

In [ ]:
%%writefile /tmp/vb/src/pipeline.py
"""Resumable band pipeline: stage inputs -> CUDA spiral fit -> render every winding -> ink (ink_9um
seed42+43, both directions, optional Hecate) -> void-masked montages. All persistent state lives in
OUT=/kaggle/working/<tag>; re-running (same session, or a new one with the previous output attached
as an input and P["resume_from"] pointing at it) skips finished work.

    python pipeline.py params.json
"""
import glob, json, os, queue, re, shutil, signal, subprocess, sys, threading, time
import numpy as np

P = json.load(open(sys.argv[1]))
T_START = float(P.get("t_start", time.time()))
BUDGET = float(P.get("max_hours", 2.8)) * 3600
OUT = P["out"]; TMP = P["tmp"]; VILLA = P["villa"]; PY = P.get("fit_python", sys.executable)
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
os.environ.setdefault("CHUNK_CACHE", os.path.join(TMP, "chunks"))
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import fetch, render_gpu, montage  # noqa: E402

LOG = open(os.path.join(OUT, "log.txt"), "a")
# ink passes: "screen" = seed42 alone at 25% overlap (~4.5x cheaper, both directions kept: the fitted
# meshes' recto side is not known in advance); "full" = seed42+43 ensemble at 50% overlap (the validated
# local recipe), for the control and for confirming candidate windings found by a screen.
INK_MODE = P.get("ink_mode") or ("full" if P["mode"] == "control" else "screen")
PRED = os.path.join(OUT, "pred" if INK_MODE == "full" else "pred_screen")
DONE = os.path.join(OUT, "windings.json" if INK_MODE == "full" else "windings_screen.json")


def log(*a):
    s = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a)
    print(s, flush=True); LOG.write(s + "\n"); LOG.flush()


def left():
    return BUDGET - (time.time() - T_START)


def state(**kw):
    p = os.path.join(OUT, "state.json")
    s = json.load(open(p)) if os.path.exists(p) else {}
    s.update(kw)
    json.dump(s, open(p, "w"), indent=1)
    return s


def resume_copy():
    src = P.get("resume_from")
    if not src:
        return
    cand = [d for d in glob.glob(os.path.join(src, "**", "state.json"), recursive=True)
            if os.path.basename(os.path.dirname(d)) == os.path.basename(OUT)]
    if not cand:
        log("resume_from given but no matching state.json under", src); return
    d = os.path.dirname(cand[0])
    fitted = bool(find_meshes(os.path.join(d, "runs"))[1])
    for root, _, files in os.walk(d):
        for f in files:
            if fitted and f.endswith(".ckpt"):
                continue  # meshes exist: the fit is not resumed, so its checkpoint is not needed
            s = os.path.join(root, f); t = os.path.join(OUT, os.path.relpath(s, d))
            if not os.path.exists(t):
                os.makedirs(os.path.dirname(t), exist_ok=True); shutil.copy2(s, t)
    log("resumed state from", d)


def find_meshes(root):
    """(run_dir, [winding dirs]) of the newest fit under root holding plain wNNN tifxyz windings.
    The exporter (spiral_helpers.save_mesh) writes <run>/meshes/<name>[_<tag>]/wNNN[_spliced][_<tag>];
    keep the plain (non-spliced) windings in any such layout."""
    best = None
    for dp, dns, fns in os.walk(root):
        ws = sorted(os.path.join(dp, d) for d in dns
                    if re.match(r"w\d{3}(_|$)", d) and "_spliced" not in d and os.path.exists(os.path.join(dp, d, "x.tif")))
        if ws:
            run = dp.split(os.sep + "meshes" + os.sep)[0] if os.sep + "meshes" + os.sep in dp + os.sep else dp
            key = (os.path.getmtime(dp), len(ws))
            if best is None or key > best[0]:
                best = (key, run, ws)
            dns[:] = []
    return (best[1], best[2]) if best else (None, [])


# ---------------------------------------------------------------- inputs + fit
def stage_inputs(vol_name, voxel):
    scroll, ts = P["scroll"], vol_name.split("-")[0]
    ds = os.path.join(TMP, "dataset")
    os.makedirs(ds, exist_ok=True)
    log("tracks index + crossings ...")
    store, dbm = fetch.stage_tracks(scroll, ts, ds)
    z0, z1 = P.get("z0"), P.get("z1")
    if z0 in (None, "auto"):
        z0, z1, npts = fetch.auto_band(store, int(P.get("band_height", 1000)))
        log(f"auto band [{z0},{z1}) ({npts / 1e6:.1f}M track points)")
    z0, z1 = int(z0), int(z1)
    state(z0=z0, z1=z1)
    n = fetch.stage_band_coords(store, f"{fetch.DL}/{scroll}/{ts}/tracks/{dbm}", z0, z1)
    log(f"band tracks: {n}")
    las = fetch.stage_lasagna(scroll, ts, ds, z0, z1) if P.get("use_lasagna", True) else None
    log("lasagna:", "none" if not las else f"group {las[0]} x{las[1]}")
    umb = fetch.published_umbilicus(scroll, ts)
    src = "published"
    if umb is None and las:
        umb = {"control_points": fetch.lasagna_umbilicus(las[2], las[0], las[1], z0, z1)}; src = "lasagna-estimate"
    if umb is None or not umb.get("control_points"):
        umb = {"control_points": fetch.ct_umbilicus(f"{fetch.S3}/{scroll}/volumes/{vol_name}", z0, z1)}; src = "ct-centroid"
    json.dump(umb, open(os.path.join(ds, "umbilicus.json"), "w"), indent=1)
    shutil.copy(os.path.join(ds, "umbilicus.json"), os.path.join(OUT, f"umbilicus_{src}.json"))
    pts = [p for p in umb["control_points"] if z0 - 500 <= p["z"] <= z1 + 500]
    log(f"umbilicus ({src}): {len(umb['control_points'])} pts, band pts:",
        [(round(p['z']), round(p['x']), round(p['y'])) for p in pts[:6]])
    spec = fetch.write_spec(ds, scroll, voxel, P.get("sense", "CW"), dbm, las)
    shutil.copy(os.path.join(ds, "spiral-scroll.json"), os.path.join(OUT, "spiral-scroll.json"))
    if las:
        log("packing lasagna resident pools ...")
        r = subprocess.run([PY, "pack_resident_pools.py", os.path.join(ds, "lasagna_inputs"), "--what", "normals,grad_mag",
                            "--normal-group", las[0], "--io-threads", "8", "--force"],
                           cwd=os.path.join(VILLA, "spiral-fitting"), capture_output=True, text=True)
        log(r.stdout[-1500:], r.stderr[-1500:])
        if r.returncode:
            raise SystemExit("pack_resident_pools failed")
    return ds, z0, z1, bool(las)


def fit(ds, z0, z1, has_las):
    runs = os.path.join(OUT, "runs")
    tag = P.get("tag", "band")
    ov = {"z_begin": z0, "z_end": z1, "input_use_verified_patches": False, "input_use_tracks": True,
          "input_use_normals": has_las, "input_use_gradient_magnitude": has_las,
          "input_use_winding_inference": False, "input_use_outer_shell": False, "input_use_fibers": False,
          "input_use_pcl_absolute": False, "input_use_pcl_relative": False, "input_use_pcl_same_winding": False,
          "input_use_pcl_drawn_control_points": False, "dense_spacing_mode": "grad_mag",
          "loss_weight_shell_outer": 0, "optimizer_num_training_steps": int(P.get("steps", 3000))}
    if not has_las:
        ov["loss_weight_dense_spacing"] = 0
    ov.update(P.get("fit_overrides", {}))
    env = dict(os.environ, FIT_SPIRAL_DEVICE="cuda", FIT_SPIRAL_CACHE_DIR=os.path.join(TMP, "cache_spiral"),
               FIT_SPIRAL_OUT_DIR=runs, FIT_SPIRAL_RUN_TAG=tag, FIT_SPIRAL_CONFIG_OVERRIDES=json.dumps(ov),
               WANDB_MODE="disabled", PYTHONUNBUFFERED="1")
    if P.get("float32", True):
        env["FIT_SPIRAL_MAX_PRECISION_FLOAT"] = "32"
    # preflight: the fit must load the band from the packed .vctracks store; the .dbm is only a stub
    dbm = json.load(open(os.path.join(ds, "spiral-scroll.json")))["paths"]["tracks_dbm"]
    chk = (f"import tracks; c = tracks._load_packed_track_collection({os.path.join(ds, dbm)!r}, {z0}, {z1}); "
           f"assert c is not None and len(c) > 0, 'packed track store not usable'; print('packed store ok:', len(c), 'tracks')")
    r = subprocess.run([PY, "-c", chk], cwd=os.path.join(VILLA, "spiral-fitting"), capture_output=True, text=True)
    log(r.stdout[-800:], r.stderr[-2500:])
    if r.returncode:
        raise SystemExit("packed track store preflight failed (see above)")
    triton = P.get("triton", True)
    for attempt in range(2):
        prev = sorted(d for d in glob.glob(os.path.join(runs, f"*_{tag}")) if os.path.exists(os.path.join(d, "checkpoint_fitted.ckpt")))
        e = dict(env)
        if prev:
            e.update(FIT_SPIRAL_RUN_DIR=prev[-1], FIT_SPIRAL_RESUME_PATH=os.path.join(prev[-1], "checkpoint_fitted.ckpt"))
            log("resuming fit from", prev[-1])
        if not triton:
            e["FIT_SPIRAL_TRITON"] = "0"
        logf = os.path.join(OUT, f"fit_{tag}.log")
        log(f"fit [{z0},{z1}) steps={ov['optimizer_num_training_steps']} triton={triton} -> {logf}")
        with open(logf, "a") as lf:
            pr = subprocess.Popen([PY, "fit_spiral.py", "--dataset", ds], cwd=os.path.join(VILLA, "spiral-fitting"),
                                  env=e, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
            reserve = float(P.get("reserve_hours_for_ink", 1.0)) * 3600
            while pr.poll() is None:
                time.sleep(20)
                if left() < reserve:
                    log("fit out of budget: stopping (resume next session from the 1000-step autosave)")
                    os.killpg(pr.pid, signal.SIGINT); time.sleep(60)
                    if pr.poll() is None:
                        os.killpg(pr.pid, signal.SIGKILL)
                    state(fit="incomplete"); return None
        tail = open(logf, errors="replace").read()[-6000:]
        run_dir, meshes = find_meshes(runs)
        if pr.returncode == 0 and meshes:
            state(fit="done", run_dir=os.path.relpath(run_dir, OUT)); log(f"fit done: {len(meshes)} windings in {run_dir}")
            return run_dir
        if pr.returncode == 0:
            log("fit exited 0 but no wNNN meshes found; runs tree:\n" + "\n".join(
                os.path.relpath(dp, runs) for dp, _, _ in os.walk(runs) if dp.count(os.sep) - runs.count(os.sep) <= 3))
        log(f"fit exited {pr.returncode}; tail:\n{tail[-2500:]}")
        if triton and re.search(r"triton|ptxas|CompilationError|PTX", tail, re.I):
            triton = False; state(triton=False); continue
        raise SystemExit("fit failed")
    raise SystemExit("fit failed twice")


# ---------------------------------------------------------------- render + ink
def winding_loop(items, vol_url, level, frame_level, voxel_um):
    """items: [(name, tifxyz_dir)]. Renders in a background thread while the GPU runs ink."""
    import torch
    from ink import Ink, hecate
    pred = PRED; rend = os.path.join(OUT, "renders"); os.makedirs(pred, exist_ok=True); os.makedirs(rend, exist_ok=True)
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    todo = [(n, d) for n, d in items if n not in done]
    log(f"windings: {len(items)} total, {len(todo)} to do")
    if not todo:
        return
    ngpu = torch.cuda.device_count()
    rdev = f"cuda:{ngpu - 1}" if ngpu else "cpu"
    cc = torch.cuda.get_device_capability(0) if ngpu else (0, 0)
    amp = P.get("amp") or ("fp16" if cc < (8, 0) else "auto")
    ck = P["ink_checkpoints"] if INK_MODE == "full" else P["ink_checkpoints"][:1]
    overlap = float(P.get("ink_overlap") or (0.5 if INK_MODE == "full" else 0.25))
    ink = Ink(ck[0], ck[1:], gpus=",".join(map(str, range(ngpu))) if ngpu > 1 else "", amp=amp,
              batch=int(P.get("ink_batch", 16)), overlap=overlap)
    log(f"ink ready ({INK_MODE}): {len(ck)} checkpoint(s), overlap {overlap}, gpus={ngpu}, amp={amp}, render on {rdev}")
    q = queue.Queue(maxsize=1)
    stop = threading.Event()
    per = []

    def producer():
        for n, d in todo:
            if stop.is_set():
                break
            t0 = time.time()
            z = os.path.join(TMP, "renders", f"{n}.zarr")
            try:
                cv, vm = render_gpu.render(d, vol_url, z, layers=int(P.get("layers", 31)), level=level,
                                           flip=bool(P.get("flip", True)), frame_level=frame_level, device=rdev, log=log)
            except Exception as e:
                log(f"render fail {n}: {e!r}"); q.put((n, None, 0, 0, 0)); continue
            L = cv.shape[0]
            mid = cv[L // 2 - 6:L // 2 + 7].max(0)
            import cv2
            cv2.imwrite(os.path.join(rend, f"{n}_max.png"), mid)
            cover = float(((cv[L // 2] > 0) & vm).sum() / max(1, vm.sum()))
            area = float(vm.sum()) * (voxel_um * 2 ** level * 1e-4) ** 2
            del cv
            q.put((n, z, cover, area, time.time() - t0))
            render_gpu.prune_cache(os.environ["CHUNK_CACHE"], float(P.get("cache_gb", 25)))
        q.put(None)
    th = threading.Thread(target=producer, daemon=True); th.start()
    summary_p = os.path.join(OUT, f"windings_summary{'' if INK_MODE == 'full' else '_screen'}.json")
    summary = json.load(open(summary_p)) if os.path.exists(summary_p) else {}
    while True:
        it = q.get()
        if it is None:
            break
        n, z, cover, area = it[:4]
        if z is None:
            continue
        t0 = time.time()
        if cover < float(P.get("min_cover", 0.15)):
            log(f" {n}: papyrus cover {cover:.2f} < min -> skip ink")
        else:
            ink(z, os.path.join(pred, f"{n}.tif"))
            if P.get("hecate"):
                for rev in (False, True):
                    r = hecate(P["hecate_py"], P["hecate_ck"], z, os.path.join(pred, f"{n}_hecate{'_reverse' if rev else ''}.png"),
                               voxel_um * 2 ** level, reverse=rev, device="cuda:0")
                    if r.returncode:
                        log(" hecate fail", r.stderr[-800:])
        shutil.rmtree(z, ignore_errors=True)
        dt = time.time() - t0 + it[4]
        per.append(dt)
        done.add(n); json.dump(sorted(done), open(DONE, "w"))
        summary[n] = {"cover": round(cover, 3), "area_cm2": round(area, 2), "sec": round(dt)}
        json.dump(summary, open(summary_p, "w"), indent=1)
        log(f" {n}: cover {cover:.2f}, {area:.1f} cm2, {dt:.0f}s; {len(done)}/{len(items)} done, {left() / 60:.0f} min left")
        if left() < 1.5 * np.median(per[-5:]) + 600:
            log("time budget reached: stopping windings (rerun with resume to continue)")
            stop.set()
            while q.get() is not None:
                pass
            break
    th.join(timeout=5)


def montages(name):
    pred = PRED; rend = os.path.join(OUT, "renders"); mdir = os.path.join(OUT, "montage")
    name = f"{name}_{INK_MODE}"
    os.makedirs(mdir, exist_ok=True)
    for d in ("forward", "reverse"):
        outs = montage.montage(pred, rend, d, os.path.join(mdir, f"{name}_{d}"), width=int(P.get("montage_width", 3000)))
        log(f"montage {d}: {len(outs)} sheets")


def main():
    resume_copy()
    vol_name, voxel = fetch.resolve_volume(P["scroll"], P["volume"])
    vol_url = f"{fetch.S3}/{P['scroll']}/volumes/{vol_name}"
    level = int(P.get("level", 0))
    log(f"== {P['mode']} {P['scroll']} {vol_name} ({voxel} um, render level {level} = {voxel * 2 ** level:.2f} um); "
        f"budget {BUDGET / 3600:.1f} h")
    state(mode=P["mode"], scroll=P["scroll"], volume=vol_name)
    if P["mode"] == "control":
        seg = P.get("segment", "20240304144031-w018_20240304144031_flatboi")
        mesh = P.get("mesh", "20240304144031-on-20251217075048-2.399um.tifxyz")
        d = os.path.join(TMP, "control", "w018")
        os.makedirs(d, exist_ok=True)
        for f in ("meta.json", "x.tif", "y.tif", "z.tif"):
            fetch.download(f"{fetch.S3}/{P['scroll']}/segments/{seg}/mesh/{mesh}/{f}", os.path.join(d, f), threads=8, part=16 << 20)
        ref = [k for k in fetch.s3_list(f"{P['scroll']}/segments/{seg}/ink-detection/downsampled/")[1] if "2.399um" in k]
        if ref:
            fetch.download(f"{fetch.S3}/{ref[0]}", os.path.join(OUT, "official_2.4um_ink_ds8.jpg"))
        winding_loop([("w018", d)], vol_url, level, int(P.get("frame_level", 0)), voxel)
        montages("control")
        for suf in ("", "_reverse"):
            f = os.path.join(PRED, f"w018{suf}.tif")
            if os.path.exists(f):
                montage.masked_fullres(f, os.path.join(OUT, "renders", "w018_max.png"), os.path.join(OUT, f"w018{suf}_{INK_MODE}_masked.jpg"))
        log("control done"); return
    run, ws = find_meshes(os.path.join(OUT, "runs"))
    if ws:
        log(f"fit already done: {len(ws)} windings in {run} -> skipping inputs + fit")
    else:
        ds, z0, z1, has_las = stage_inputs(vol_name, voxel)
        log(f"inputs staged, {left() / 60:.0f} min left")
        run = fit(ds, z0, z1, has_las)
        if run is None:
            return
        run, ws = find_meshes(os.path.join(OUT, "runs"))
    lo, hi = P.get("windings", [0, 9999])
    items = [(os.path.basename(w)[:4], w) for w in ws if lo <= int(os.path.basename(w)[1:4]) <= hi]
    if P.get("winding_list"):  # e.g. ["w052", "w053"]: confirm only these (typically with ink_mode="full")
        items = [t for t in items if t[0] in set(P["winding_list"])]
    # outer windings first (largest area per GPU-minute; inner turns are often crushed)
    items.sort(key=lambda t: -int(t[0][1:]))
    winding_loop(items, vol_url, level, 0, voxel)
    montages(P["scroll"])
    state(last_finished=time.strftime("%Y-%m-%d %H:%M"))
    log("band done for this session")


if __name__ == "__main__":
    main()

In [ ]:
open(f"{TMP}/smoke.py", "w").write(SMOKE)
sh(f"cd {TMP} && PYTHONPATH={VILLA}/vesuvius/src {VINK}/bin/python {TMP}/smoke.py")
P = dict(RUN, out=f"{W}/{RUN['tag']}", tmp=f"{TMP}/{RUN['tag']}", villa=VILLA, fit_python=f"{VFIT}/bin/python", t_start=T0,
         ink_checkpoints=[f"{TMP}/ck/ink_9um/hybrid_3d2d-seed42/step-075000.pth", f"{TMP}/ck/ink_9um/hybrid_3d2d-seed43/step-075000.pth"],
         hecate_py=f"{TMP}/ck/hecate/hecate.py", hecate_ck=f"{TMP}/ck/hecate/hecate_9.6um.pth")
json.dump(P, open(f"{TMP}/params.json", "w"))
env = dict(os.environ, PYTHONPATH=f"{VILLA}/vesuvius/src", CHUNK_CACHE=f"{TMP}/chunks", OMP_NUM_THREADS="4")
p = subprocess.Popen([f"{VINK}/bin/python", "-u", f"{TMP}/src/pipeline.py", f"{TMP}/params.json"], env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "\r" not in line: print(line, end="")
print("exit", p.wait(), f"total {(time.time() - T0) / 60:.1f} min")

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob(f"{W}/{RUN['tag']}/montage/*.jpg"))[:6] + sorted(glob.glob(f"{W}/{RUN['tag']}/*_masked.jpg")):
    print(f); display(Image(f, width=1200))
sh(f"du -sh {W}/{RUN['tag']}; ls {W}/{RUN['tag']}")